In [89]:
"""
# META DATA - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

    Developer Details: 
        Name: Swapnendu and Aryansh
        Role: Developers
        Code ownership rights: Swapnendu and Aryansh

    Version:
        Version: V 1.1 (23rd November, 2024)
            Developer: Swapnendu and Aryansh
            Reviewer: Vansh Raja
            Unit test: Pass
            Integration test: Pass
     
    Description: This script demonstrates basic PySpark operations, including reading data, 
    data exploration, handling missing values, filter operations, group-by aggregations, 
    join operations, and handling date-related issues by converting strings to date data types.

# CODE - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

    Dependency: 
        Environment:     
            Python: 3.10.12
            PySpark: 3.5.3
            Numpy: 2.1.3
            
    DATASETS USED:
        Dataset 1:
            Name: pyspark_order_table_modified
            Path: /Data/pyspark_order_table_modified.csv
            Description: Order data with customer and product information.
        Dataset 2:
            Name: pyspark_customer_table
            Path: /Data/pyspark_customer_table.csv
            Description: Customer information.

    OPERATIONS CARRIED OUT:
        1. Reading data from CSV files.
        2. Data exploration (schema, data types, descriptive statistics).
        3. Handling missing values (dropping, filling, imputation).
        4. Filter operations (using `where` and `filter` clauses).
        5. Group-by aggregations (sum, max, mean).
        6. Join operations (inner, left, right, outer).
        7. Handling date-related issues (converting strings to date data types).
"""


'\n# META DATA - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - \n\n    Developer Details: \n        Name: Swapnendu and Aryansh\n        Role: Developers\n        Code ownership rights: Swapnendu and Aryansh\n\n    Version:\n        Version: V 1.1 (23rd November, 2024)\n            Developer: Swapnendu and Aryansh\n            Reviewer: Vansh Raja\n            Unit test: Pass\n            Integration test: Pass\n     \n    Description: This script demonstrates basic PySpark operations, including reading data, \n    data exploration, handling missing values, filter operations, group-by aggregations, \n    join operations, and handling date-related issues by converting strings to date data types.\n\n# CODE - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - \n\n    Dependency: \n        Environment:     \n            Python: 3.10.12\n            PySpark: 3.5.3\n            Numpy: 2.1.3\n      

In [32]:
import pyspark 

## Some basic operations we will cover

- PySpark Dataframe
- Reading the Dataset
- Checking DataTypes of the Columns(Schema)
- Selecting Columns and Indexing
- Checking Describe option similar to Pandas
- Adding Columns
- Dropping Columns
- Renaming Columns

### 1. Creating a PySpark Dataframe
First, we have to create a spark session, takes some time for the first time




In [33]:
# This step helps to ensure that the proper java version is set on your notebook.
# Before this make sure that pyspark and java 11 is installed on your machine
import os

# # Set JAVA_HOME and PATH for Spark (on Windows)
# os.environ['JAVA_HOME'] = r'C:\Program Files\OpenJDK\openjdk-11'  # Replace this with your actual JDK installation path
# os.environ['PATH'] = os.environ['JAVA_HOME'] + r'\bin;' + os.environ['PATH']

# Set JAVA_HOME and PATH for MACOS
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@11'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']


In [34]:
# Just to check if it properly worked
!java -version

openjdk version "11.0.25" 2024-10-15
OpenJDK Runtime Environment Homebrew (build 11.0.25+0)
OpenJDK 64-Bit Server VM Homebrew (build 11.0.25+0, mixed mode)


In [35]:
from pyspark.sql import SparkSession

# Stop the existing Spark session (if any)
spark = SparkSession.builder.getOrCreate()
spark.stop()

# spark = SparkSession.builder.appName("pyspark-session").getOrCreate()
spark = SparkSession.builder.appName("Test").getOrCreate()

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

print("Spark session created successfully")

Spark session created successfully


In [36]:
spark ## Metadata related to spark instance created.

### 2. Reading the Dataset
- With headers enabled --> option('header','true')
- To avoid getting all columns by default as string --> csv ( inferSchema = True) 
- Top 5 rows --> show()

In [37]:
# Here input the path to pyspark_order_table_modified.csv from your Data folder. You can right click to copy it.
dataset_path = input("Enter the path to the dataset: ")
df_pyspark=spark.read.csv(dataset_path , header=True, inferSchema=True)
## creating a pyspark dataframe

In [38]:
df_pyspark.show(5)
# show() is equivalent to head() from pandas.
# A better structered manner to visualize data from dataframe.

+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
|         68|       1|   Product A|      NULL|           20|      42|         840|North America|  Jane Smith|
|          0|       2|   Product B| 4/14/2021|           30|      81|        2430|    Australia|  Jane Smith|
|         93|       3|   Product A|      NULL|           20|      68|        1360|North America|Mike Johnson|
|         43|       4|   Product C|11/22/2022|           40|      56|        NULL|       Europe|  Jane Smith|
|         69|       5|   Product C|      NULL|           40|      83|        3320|         Asia|  Jane Smith|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
only showi

### 3. Getting the datatypes of columns --> .printSchema(), .dtypes


In [39]:
df_pyspark.printSchema() ## df Schema --> dataType(if nullable or not)

root
 |-- customer_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- product_price: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)



In [40]:
df_pyspark.dtypes ## gives a list of types of col names, data type of that col.

[('customer_id', 'int'),
 ('order_id', 'int'),
 ('product_name', 'string'),
 ('order_date', 'string'),
 ('product_price', 'int'),
 ('quantity', 'int'),
 ('order_amount', 'int'),
 ('region', 'string'),
 ('sales_rep', 'string')]

In [41]:
type(df_pyspark)
# datatype of the dataframe

pyspark.sql.dataframe.DataFrame

### 4. Checking Columns and Indexing

In [42]:
# Get the column names
df_pyspark.columns 

['customer_id',
 'order_id',
 'product_name',
 'order_date',
 'product_price',
 'quantity',
 'order_amount',
 'region',
 'sales_rep']

In [43]:
#  Select ONE particular column or querying ONE column
df_pyspark.select('customer_id')
# Return type is dataframe.

DataFrame[customer_id: int]

In [44]:
df_pyspark.select('customer_id').show(5) ## Querying onto 1 column. Using SELECT

+-----------+
|customer_id|
+-----------+
|         68|
|          0|
|         93|
|         43|
|         69|
+-----------+
only showing top 5 rows



In [45]:
# Selecting multiple columns, pass a list of column names.
df_pyspark.select(["order_id","customer_id","product_name"]) ## Querying onto multiple columns. 
# Returns dataframe

DataFrame[order_id: int, customer_id: int, product_name: string]

In [46]:
df_pyspark.select(["order_id","customer_id","product_name"]).show(5) 

+--------+-----------+------------+
|order_id|customer_id|product_name|
+--------+-----------+------------+
|       1|         68|   Product A|
|       2|          0|   Product B|
|       3|         93|   Product A|
|       4|         43|   Product C|
|       5|         69|   Product C|
+--------+-----------+------------+
only showing top 5 rows



### 5. Using describe

In [47]:
# Without passing any list, describes whole df
df_pyspark.describe().show() ## gives a statistical description column-wise of the dataframe.

+-------+-----------------+-----------------+------------+----------+-----------------+------------------+------------------+-------------+------------+
|summary|      customer_id|         order_id|product_name|order_date|    product_price|          quantity|      order_amount|       region|   sales_rep|
+-------+-----------------+-----------------+------------+----------+-----------------+------------------+------------------+-------------+------------+
|  count|             1000|             1000|         922|       712|              922|               983|               867|          865|         980|
|   mean|           52.012|            500.5|        NULL|      NULL|30.04338394793926| 51.06307222787385|1539.6424452133795|         NULL|        NULL|
| stddev|28.44202532881237|288.8194360957494|        NULL|      NULL|8.111483304060648|29.020010118763715| 994.7034032612953|         NULL|        NULL|
|    min|                0|                1|   Product A|  1/1/2020|             

In [48]:
#Passing list of relevant column names
df_pyspark.describe(["product_name","order_amount","quantity","region","sales_rep"]).show()

+-------+------------+------------------+------------------+-------------+------------+
|summary|product_name|      order_amount|          quantity|       region|   sales_rep|
+-------+------------+------------------+------------------+-------------+------------+
|  count|         922|               867|               983|          865|         980|
|   mean|        NULL|1539.6424452133795| 51.06307222787385|         NULL|        NULL|
| stddev|        NULL| 994.7034032612953|29.020010118763715|         NULL|        NULL|
|    min|   Product A|                20|                 1|         Asia|  Jane Smith|
|    max|   Product C|              4000|               100|North America|Mike Johnson|
+-------+------------+------------------+------------------+-------------+------------+



### 6. Adding and Dropping Columns [ NOT INPLACE, NEEDS REASSIGNMENT OR NEW ASSIGNMENT]

In [49]:
# Adding column

altered_df=df_pyspark.withColumn("New order_amount" , df_pyspark[col:="order_amount"]) 
## Syntax: 'column name','data to be in the column{here the same data as order_amount}'

In [50]:
altered_df.show(5)

+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+----------------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|New order_amount|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+----------------+
|         68|       1|   Product A|      NULL|           20|      42|         840|North America|  Jane Smith|             840|
|          0|       2|   Product B| 4/14/2021|           30|      81|        2430|    Australia|  Jane Smith|            2430|
|         93|       3|   Product A|      NULL|           20|      68|        1360|North America|Mike Johnson|            1360|
|         43|       4|   Product C|11/22/2022|           40|      56|        NULL|       Europe|  Jane Smith|            NULL|
|         69|       5|   Product C|      NULL|           40|      83|        3320|         Asia|  Jane Smith|  

In [51]:
# Dropping a column

altered_df=altered_df.drop(col:="New order_amount")
altered_df.show(5)

+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
|         68|       1|   Product A|      NULL|           20|      42|         840|North America|  Jane Smith|
|          0|       2|   Product B| 4/14/2021|           30|      81|        2430|    Australia|  Jane Smith|
|         93|       3|   Product A|      NULL|           20|      68|        1360|North America|Mike Johnson|
|         43|       4|   Product C|11/22/2022|           40|      56|        NULL|       Europe|  Jane Smith|
|         69|       5|   Product C|      NULL|           40|      83|        3320|         Asia|  Jane Smith|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
only showi

### 7. Renaming a column (NOT INPLACE)

In [52]:
altered_df.withColumnRenamed("customer_id","Id of customer").show(5)
## Syntax: 'old name', 'new name'

+--------------+--------+------------+----------+-------------+--------+------------+-------------+------------+
|Id of customer|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|
+--------------+--------+------------+----------+-------------+--------+------------+-------------+------------+
|            68|       1|   Product A|      NULL|           20|      42|         840|North America|  Jane Smith|
|             0|       2|   Product B| 4/14/2021|           30|      81|        2430|    Australia|  Jane Smith|
|            93|       3|   Product A|      NULL|           20|      68|        1360|North America|Mike Johnson|
|            43|       4|   Product C|11/22/2022|           40|      56|        NULL|       Europe|  Jane Smith|
|            69|       5|   Product C|      NULL|           40|      83|        3320|         Asia|  Jane Smith|
+--------------+--------+------------+----------+-------------+--------+------------+-----------

## Pyspark Handling Missing Values
    
  - Dropping Columns
  - Dropping Rows
  - Various Parameters in Dropping Functionality
  - Handling Missing Values by mean, median or mode

In [53]:
# Dropping rows based on missing values

# Let's get a stat on missing val first.
# By default pyspark lacks that functionality, so lets built a function that takes param: dataframe, returns: dataframe of missing val state col-wise.


In [54]:
def getNullReport(df_pyspark):
    from pyspark.sql.functions import col, sum

    # Check nulls in each column
    #
    # 1. for c in df_pyspark.columns returns a list of column names as strings. Check : [c for c in df_pyspark.columns ]
    # 2. col(c) creates a column object out of the column name, which is used for pyspark dataframe operations. Check : [col(c) for c in df_pyspark.columns ]
    # 3. col(c).isNull() creates a column object, giving true for null values and false for non-null. Check: [col(c).isNull() for c in df_pyspark.columns ]
    # 4. cast("int") converts the boolean into int; sum(col(c).isNull().cast("int")) gives the sum of null values in a column.  Check: [sum(col(c).isNull().cast("int")) for c in df_pyspark.columns ]
    # 5. alias() is used to change the name into something sensible instead of showing "column_name is NULL" ex: customer_id IS NULL
    # 6. df_pyspark.select(null_counts) needed to be applied as null_count is a list of column object 
    # 7. select() method takes either column names or column expressions as its argument.
    #
    #
    #
    null_counts =[sum(col(c).isNull().cast("int")).alias(f"Nulls in: {c} " ) for c in df_pyspark.columns]
    null_value_report_dataframe=df_pyspark.select(null_counts)

    return null_value_report_dataframe

In [55]:
null_report_dataframe = getNullReport(df_pyspark)
null_report_dataframe.show()

+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|Nulls in: customer_id |Nulls in: order_id |Nulls in: product_name |Nulls in: order_date |Nulls in: product_price |Nulls in: quantity |Nulls in: order_amount |Nulls in: region |Nulls in: sales_rep |
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|                     0|                  0|                     78|                  288|                      78|                 17|                    133|              135|                  20|
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+



## Dropping the missing values

#### Understanding .na.drop()
how="any/all",threshold=None,subset=None

In [56]:
#### how 

## how can have 2 values 'any' and 'all'
## any : drops a row if it contains any null value
## all : dropp a row if it contains all values as null.

print("Using how='any'")
getNullReport(df_pyspark.na.drop(how="any")).show()
print("Using how='all'") # We do not have all rows as null
getNullReport(df_pyspark.na.drop(how="all")).show()


Using how='any'
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|Nulls in: customer_id |Nulls in: order_id |Nulls in: product_name |Nulls in: order_date |Nulls in: product_price |Nulls in: quantity |Nulls in: order_amount |Nulls in: region |Nulls in: sales_rep |
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|                     0|                  0|                      0|                    0|                       0|                  0|                      0|                0|                   0|
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+-----------

In [57]:
##### threshold

## thersh = n; atleast n columns need to be non null

getNullReport(df_pyspark.na.drop(thresh=7)).show()

+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|Nulls in: customer_id |Nulls in: order_id |Nulls in: product_name |Nulls in: order_date |Nulls in: product_price |Nulls in: quantity |Nulls in: order_amount |Nulls in: region |Nulls in: sales_rep |
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|                     0|                  0|                      0|                  258|                       0|                 11|                     46|              116|                  16|
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+



In [58]:
##### subset

## subset = [list of column names] drops null values from those columns that have been mentioned.

getNullReport(df_pyspark.na.drop(subset=["order_amount","sales_rep"])).show()

# Consider a case where we need to find the name of the sales rep that got us the max(order_amount)
# thus we wont be needing column that has no sales_rep

+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|Nulls in: customer_id |Nulls in: order_id |Nulls in: product_name |Nulls in: order_date |Nulls in: product_price |Nulls in: quantity |Nulls in: order_amount |Nulls in: region |Nulls in: sales_rep |
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|                     0|                  0|                      0|                  246|                       0|                  0|                      0|              110|                   0|
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+



## Filling the missing values


#### Understanding .na.fill()
value,subset : params

In [59]:
##### value = ?; fills all the missing values with ?
##### susbet; only allows that operation to be carried on certain columns

getNullReport(df_pyspark).show()

+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|Nulls in: customer_id |Nulls in: order_id |Nulls in: product_name |Nulls in: order_date |Nulls in: product_price |Nulls in: quantity |Nulls in: order_amount |Nulls in: region |Nulls in: sales_rep |
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|                     0|                  0|                     78|                  288|                      78|                 17|                    133|              135|                  20|
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+



In [60]:
getNullReport(df_pyspark.na.fill(value="Missing sales rep", subset=["sales_rep"])).show()
altered_df=df_pyspark.na.fill(value="Missing sales rep", subset=["sales_rep"])

altered_df.select(["order_id","sales_rep"]).where(altered_df["sales_rep"]=="Missing sales rep").show(5)


+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|Nulls in: customer_id |Nulls in: order_id |Nulls in: product_name |Nulls in: order_date |Nulls in: product_price |Nulls in: quantity |Nulls in: order_amount |Nulls in: region |Nulls in: sales_rep |
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+
|                     0|                  0|                     78|                  288|                      78|                 17|                    133|              135|                   0|
+----------------------+-------------------+-----------------------+---------------------+------------------------+-------------------+-----------------------+-----------------+--------------------+

+---

## Handling missing values with Imputer Functions

IMPUTER function,  lets you replace NULL values in a string or numerical expression. You can replace NULL values with the most frequently used value for string expressions, or the mean or median value for numerical expressions.

In [61]:
df_pyspark.show(5)

+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
|         68|       1|   Product A|      NULL|           20|      42|         840|North America|  Jane Smith|
|          0|       2|   Product B| 4/14/2021|           30|      81|        2430|    Australia|  Jane Smith|
|         93|       3|   Product A|      NULL|           20|      68|        1360|North America|Mike Johnson|
|         43|       4|   Product C|11/22/2022|           40|      56|        NULL|       Europe|  Jane Smith|
|         69|       5|   Product C|      NULL|           40|      83|        3320|         Asia|  Jane Smith|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
only showi

Choice of Strategy:
- Mean: Puts the average.
- Median: Puts the middle value.
- Mode: Puts the most frequent value.

Issues with the strategies:
- Mean: Sensitive to outliers, can make dataset skewed easily.
- Mode: In unbalanced dataset, can lead to bias as placed value is most frequent one.
- Median: Can reduce variation in dataset.

In [63]:
from pyspark.ml.feature import  Imputer

imputer = Imputer(inputCols=["product_price","quantity","order_amount"]
, outputCols=[f"{c}_imputted" for c in ["product_price","quantity","order_amount"]]).setStrategy("median")

In [64]:
copy_df = df_pyspark
# Creating a copy dataframe

In [65]:
imputer.fit(copy_df).transform(copy_df).show(5)
#  Extra columns will be created.

+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+----------------------+-----------------+---------------------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|product_price_imputted|quantity_imputted|order_amount_imputted|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+----------------------+-----------------+---------------------+
|         68|       1|   Product A|      NULL|           20|      42|         840|North America|  Jane Smith|                    20|               42|                  840|
|          0|       2|   Product B| 4/14/2021|           30|      81|        2430|    Australia|  Jane Smith|                    30|               81|                 2430|
|         93|       3|   Product A|      NULL|           20|      68|        1360|North America|Mike Johnson|                    20|   

There are other techniques like KNNImputer, Supervised prediction, Unsupervised Prediction

## Filter Operations

- &, | , ==
- ~

In [66]:
##### Filter : 


# Using a where clause

copy_df.select(["order_id","customer_id","product_name"]).where(copy_df["product_name"]=="Product A").show(5)



+--------+-----------+------------+
|order_id|customer_id|product_name|
+--------+-----------+------------+
|       1|         68|   Product A|
|       3|         93|   Product A|
|       8|         80|   Product A|
|      12|         64|   Product A|
|      13|         81|   Product A|
+--------+-----------+------------+
only showing top 5 rows



In [67]:
# Using a filter clause, it takes away the necessity to use a where clause

# filter() is more Pythonic, while where() is more SQL-like.

copy_df.filter("product_name == 'Product A'")

# returns a dataframe.

DataFrame[customer_id: int, order_id: int, product_name: string, order_date: string, product_price: int, quantity: int, order_amount: int, region: string, sales_rep: string]

In [68]:
copy_df.filter("product_name == 'Product A' AND  quantity<50 ").show(1) # Pythonish
# To avoid the string issue in filter, we can also do
copy_df.filter((copy_df["product_name"]=="Product A") & (copy_df["quantity"]<50)).show(1) # SQLish

+-----------+--------+------------+----------+-------------+--------+------------+-------------+----------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|       region| sales_rep|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+----------+
|         68|       1|   Product A|      NULL|           20|      42|         840|North America|Jane Smith|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+----------+
only showing top 1 row

+-----------+--------+------------+----------+-------------+--------+------------+-------------+----------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|       region| sales_rep|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+----------+
|         68|       1|   Product A|      NULL|           20|      42|         840|North America|Jane Smith|
+---

In [69]:
#  ~ enables us to do NOT operation
copy_df.filter(~(copy_df["product_name"]=="Product A") & (copy_df["quantity"]<50)).show(1) # SQLish

copy_df.filter("product_name != 'Product A' AND  quantity<50 ").show(1)

+-----------+--------+------------+----------+-------------+--------+------------+------+------------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|region|   sales_rep|
+-----------+--------+------------+----------+-------------+--------+------------+------+------------+
|         15|       6|   Product C|11/26/2020|           40|      29|        1160|Europe|Mike Johnson|
+-----------+--------+------------+----------+-------------+--------+------------+------+------------+
only showing top 1 row

+-----------+--------+------------+----------+-------------+--------+------------+------+------------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|region|   sales_rep|
+-----------+--------+------------+----------+-------------+--------+------------+------+------------+
|         15|       6|   Product C|11/26/2020|           40|      29|        1160|Europe|Mike Johnson|
+-----------+--------+------------+----------+---

## Using Group-By and Aggregate Function

*Group-By and Aggregate Function works together*

In [70]:
#### Let's start with GroupBy.
#  I want to group-by sales_rep and lets see the net order_amount they have brought in.

copy_df.groupBy("sales_rep").sum().show()

+------------+----------------+-------------+------------------+-------------+-----------------+
|   sales_rep|sum(customer_id)|sum(order_id)|sum(product_price)|sum(quantity)|sum(order_amount)|
+------------+----------------+-------------+------------------+-------------+-----------------+
|        NULL|            1036|        10061|               540|         1107|            26980|
|Mike Johnson|           16265|       147563|              8610|        16725|           442690|
|    John Doe|           16850|       163207|              8770|        15155|           404930|
|  Jane Smith|           17861|       179669|              9780|        17208|           460270|
+------------+----------------+-------------+------------------+-------------+-----------------+



In [71]:
# GroupBy region and see which region gives max order_amount 

copy_df.groupBy("region").max().show()

+-------------+----------------+-------------+------------------+-------------+-----------------+
|       region|max(customer_id)|max(order_id)|max(product_price)|max(quantity)|max(order_amount)|
+-------------+----------------+-------------+------------------+-------------+-----------------+
|       Europe|             100|          992|                40|          100|             4000|
|         NULL|              99|         1000|                40|          100|             3960|
|North America|              99|          999|                40|          100|             3920|
|         Asia|             100|          994|                40|          100|             3960|
|    Australia|             100|          991|                40|          100|             4000|
+-------------+----------------+-------------+------------------+-------------+-----------------+



In [72]:
# GroupBy sales_rep and see which region gives mean order_amount 

copy_df.groupBy("sales_rep").mean().show()

+------------+------------------+------------------+------------------+------------------+------------------+
|   sales_rep|  avg(customer_id)|     avg(order_id)|avg(product_price)|     avg(quantity)| avg(order_amount)|
+------------+------------------+------------------+------------------+------------------+------------------+
|        NULL|              51.8|            503.05|31.764705882352942|             55.35|1587.0588235294117|
|Mike Johnson|52.131410256410255| 472.9583333333333|29.792387543252595| 54.30194805194805|1621.5750915750916|
|    John Doe| 52.49221183800623|508.43302180685356| 30.24137931034483|48.111111111111114|1488.7132352941176|
|  Jane Smith|51.472622478386164| 517.7780979827089|              30.0| 50.61176470588235|1509.0819672131147|
+------------+------------------+------------------+------------------+------------------+------------------+



In [73]:
# Can directly apply agg function too.

copy_df.agg({'order_amount':'mean'}).show()

+------------------+
| avg(order_amount)|
+------------------+
|1539.6424452133795|
+------------------+



## Using Join Operation

#### First Let us get the tables

In [77]:
ordertablePath= input("Enter the path to the order table: ")
orderTable=spark.read.csv(ordertablePath , header=True, inferSchema=True)   # order table data

customerTablePath= input("Enter the path to the customer table: ")
customerTable = spark.read.csv(customerTablePath, header=True, inferSchema=True)  # customer table data

In [78]:
orderTable.show(5)

customerTable.show(5)

+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
|         68|       1|   Product A|      NULL|           20|      42|         840|North America|  Jane Smith|
|          0|       2|   Product B| 4/14/2021|           30|      81|        2430|    Australia|  Jane Smith|
|         93|       3|   Product A|      NULL|           20|      68|        1360|North America|Mike Johnson|
|         43|       4|   Product C|11/22/2022|           40|      56|        NULL|       Europe|  Jane Smith|
|         69|       5|   Product C|      NULL|           40|      83|        3320|         Asia|  Jane Smith|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+
only showi

<img title="Types of Joins" alt="Types of Joins" src="sql joins.jpg"  width="900">

#### Let's explore each type of join

We use `.join()` Method to create Join Between 2 DataFrames

`<DF1>.join(<DF2>, < Joining Condition >, < Type Of Join : Sting>)`

In [79]:
# Inner Join

# Performing an inner join between orderTable and customerTable based on customer_id.
# The resulting DataFrame will contain only rows where customer_id matches in both tables.
# The redundant customer_id column from customerTable is dropped to avoid duplication.

innerJoin = orderTable.join(customerTable, orderTable.customer_id == customerTable.customer_id, "inner").drop(customerTable.customer_id)        # "inner" to specify Inner Join
innerJoin.show(10)

+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+-----------------+--------------------+--------------+--------------------+-------------+--------------+----------------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|    customer_name|      customer_email|customer_phone|    customer_address|customer_city|customer_state|customer_zipcode|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+-----------------+--------------------+--------------+--------------------+-------------+--------------+----------------+
|         68|       1|   Product A|      NULL|           20|      42|         840|North America|  Jane Smith|Morganica Barford|mbarford1v@ycombi...|  638-466-7873|695 Steensland Drive|     Sélestat|            C1|     67604 CEDEX|
|         93|       3|   Product A|      NULL|           20|      68|       

In [80]:
# Left Join

# Performing a left join between orderTable and customerTable based on customer_id.
# The resulting DataFrame will contain all rows from orderTable, even if there is no matching customer_id in customerTable.
# The redundant customer_id column from customerTable is dropped to avoid duplication.

leftJoin = orderTable.join(customerTable, orderTable.customer_id == customerTable.customer_id, "left").drop(customerTable.customer_id)           # "inner" to specify Left Join
leftJoin.show()

+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+-----------------+--------------------+--------------+--------------------+--------------+--------------+----------------+
|customer_id|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|    customer_name|      customer_email|customer_phone|    customer_address| customer_city|customer_state|customer_zipcode|
+-----------+--------+------------+----------+-------------+--------+------------+-------------+------------+-----------------+--------------------+--------------+--------------------+--------------+--------------+----------------+
|         68|       1|   Product A|      NULL|           20|      42|         840|North America|  Jane Smith|Morganica Barford|mbarford1v@ycombi...|  638-466-7873|695 Steensland Drive|      Sélestat|            C1|     67604 CEDEX|
|          0|       2|   Product B| 4/14/2021|           30|      81|   

In [81]:
# Right Join

# Performing a right join between orderTable and customerTable based on customer_id.
# The resulting DataFrame will contain all rows from customerTable, even if there is no matching customer_id in orderTable.
# The redundant customer_id column from orderTable is dropped to avoid duplication.

rightJoin = orderTable.join(customerTable, orderTable.customer_id == customerTable.customer_id, "right").drop(orderTable.customer_id)        # "inner" to specify Inner Join
rightJoin.show()

+--------+------------+----------+-------------+--------+------------+-------------+------------+-----------+--------------+--------------------+--------------+--------------------+-------------+--------------+----------------+
|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|customer_id| customer_name|      customer_email|customer_phone|    customer_address|customer_city|customer_state|customer_zipcode|
+--------+------------+----------+-------------+--------+------------+-------------+------------+-----------+--------------+--------------------+--------------+--------------------+-------------+--------------+----------------+
|     853|   Product A|      NULL|           20|      55|        1100|    Australia|  Jane Smith|          1|Meredeth Moors|    mmoors0@dmoz.org|  477-456-1304|193 Golden Leaf T...|       Leiden|            11|            2334|
|     801|   Product C|11/30/2020|           40|      93|        3720|North America|  Ja

In [82]:
# Full Outer Join

# Performing a full outer join between orderTable and customerTable based on customer_id.
# The resulting DataFrame will contain all rows from both orderTable and customerTable.
# If there is a matching customer_id in both tables, the corresponding rows will be joined.
# If there is no match in either table, the row will still be included with null values for the missing columns.
# The redundant customer_id column from orderTable is dropped to avoid duplication.

fullOuter = orderTable.join(customerTable, orderTable.customer_id == customerTable.customer_id, "outer").drop(orderTable.customer_id)          # "inner" to specify Inner Join
fullOuter.show()

+--------+------------+----------+-------------+--------+------------+-------------+------------+-----------+--------------+--------------------+--------------+--------------------+-------------+--------------+----------------+
|order_id|product_name|order_date|product_price|quantity|order_amount|       region|   sales_rep|customer_id| customer_name|      customer_email|customer_phone|    customer_address|customer_city|customer_state|customer_zipcode|
+--------+------------+----------+-------------+--------+------------+-------------+------------+-----------+--------------+--------------------+--------------+--------------------+-------------+--------------+----------------+
|       2|   Product B| 4/14/2021|           30|      81|        2430|    Australia|  Jane Smith|       NULL|          NULL|                NULL|          NULL|                NULL|         NULL|          NULL|            NULL|
|     316|   Product B|      NULL|           30|      63|        1890|       Europe|    

### Manipulating Date Column

Let use Get the date column aka "order_date"

In [83]:
from pyspark.sql.functions import col ## important, or else col is considered to be a previously used variable
orderTable.select(col("order_date")).show(5)

+----------+
|order_date|
+----------+
|      NULL|
| 4/14/2021|
|      NULL|
|11/22/2022|
|      NULL|
+----------+
only showing top 5 rows



In [84]:
orderTable.select("order_date").dtypes

######### Format is STRING, we need it in DATE

[('order_date', 'string')]

Custom Function to Covert From String "MM/dd/yyyy" to dateTime format

In [85]:
from pyspark.sql.functions import to_date, col, when

# Handling nulls and converting valid dates

# Adding a new column 'date_asDateTime' to store order_date as a date data type.
# This column will contain the converted date values if order_date is not null, 
# otherwise it will contain None.

df = orderTable.withColumn(
    'date_asDateTime', 
    when(col('order_date').isNotNull(), to_date(col('order_date'), 'MM/dd/yyyy')).otherwise(None)
    ## Enter the current format which is {MM/dd/yyyy}; it will convert to default {yyyy-MM-dd}
)



In [86]:
df.select("date_asDateTime").dtypes

## Data type changed to date!

[('date_asDateTime', 'date')]

In [87]:
# Show the first 5 results
df.select(col("date_asDateTime")).show(5)

+---------------+
|date_asDateTime|
+---------------+
|           NULL|
|     2021-04-14|
|           NULL|
|     2022-11-22|
|           NULL|
+---------------+
only showing top 5 rows

